<a href="https://colab.research.google.com/github/Leonardozepeda04/edt-dataa-pipeline/blob/main/notebooks/corredores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
#Cargar dataset
url_corredores = "https://raw.githubusercontent.com/Leonardozepeda04/edt-dataa-pipeline/refs/heads/main/data/raw/corredores.csv"

In [3]:
corredores = pd.read_csv(url_corredores)

In [4]:
#Exploracion de datos
corredores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_corredor        80 non-null     int64  
 1   nombre             80 non-null     object 
 2   zona               63 non-null     object 
 3   nivel              68 non-null     object 
 4   anios_experiencia  76 non-null     float64
dtypes: float64(1), int64(1), object(3)
memory usage: 3.3+ KB


In [5]:
#Limpieza de Datos Geeral
def limpiar_dataframe(df):

    df.columns = df.columns.str.strip().str.lower()

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()

    df = df.replace(r'^\s*$', pd.NA, regex=True)

    df = df.drop_duplicates()

    return df


In [6]:
#Transformacion de Datos
# Quitar espacios extra y normalizar mayúsculas/minúsculas
corredores['zona'] = corredores['zona'].astype(str).str.strip().str.title()
corredores['nivel'] = corredores['nivel'].astype(str).str.strip().str.lower()

# Convertir valores de 'nivel' a un formato uniforme (ejemplo: "junior", "mid", "senior", "elite")
niveles_map = {
    'junior': 'Junior',
    'mid': 'Mid',
    'senior': 'Senior',
    'elite': 'Elite'
}
corredores['nivel'] = corredores['nivel'].map(niveles_map)

# --- Limpieza de columna numérica ---
# Reemplazar comas por puntos (si existieran) y convertir a numérico
corredores['anios_experiencia'] = corredores['anios_experiencia'].astype(str).str.replace(",", ".")
corredores['anios_experiencia'] = pd.to_numeric(corredores['anios_experiencia'], errors='coerce')

# --- Opcional: Rellenar valores faltantes ---
corredores['zona'] = corredores['zona'].replace("nan", pd.NA)
corredores['nivel'] = corredores['nivel'].replace("nan", pd.NA)

# Ejemplo de imputación: reemplazar NaN en experiencia por 0
corredores['anios_experiencia'] = corredores['anios_experiencia'].fillna(0)



In [10]:
# Ver resultados
print(corredores.head(20))

    id_corredor                 nombre         zona   nivel  anios_experiencia
0             1      José López Flores  Paracentral     Mid                4.0
1             2      José Ortiz García       Centro  Junior                0.0
2             3     María Ramírez Cruz       Centro  Senior                6.0
3             4    Fernanda Rojas Cruz          Nan  Senior                8.0
4             5        Ana Gómez Rojas          Nan  Senior                4.0
5             6  Sofía Reyes Hernández    Occidente   Elite                3.0
6             7   Pedro Vásquez Torres        Costa     NaN                1.0
7             8  Paula Ortiz Hernández       Centro  Junior               17.0
8             9  Carlos Torres Vásquez  Paracentral  Junior                2.0
9            10     Juan Cruz Castillo    Occidente     NaN                7.0
10           11    José Morales Flores          Nan  Junior                0.0
11           12    Pedro Gómez Vásquez          Nan 

In [8]:
# --- Separar válidos y rechazados ---
validos = corredores[
    corredores['nombre'].notna() &
    corredores['zona'].notna() &
    corredores['nivel'].notna() &
    corredores['anios_experiencia'].notna()
].copy()

rechazados = corredores[
    corredores['nombre'].isna() |
    corredores['zona'].isna() |
    corredores['nivel'].isna() |
    corredores['anios_experiencia'].isna()
].copy()

In [9]:
#Motivos de rechazos
def motivo(row):
    motivos = []
    if pd.isna(row['nombre']):
        motivos.append("nombre_vacio")
    if pd.isna(row['zona']):
        motivos.append("zona_vacio")
    if pd.isna(row['nivel']):
        motivos.append("nivel_vacio")
    if pd.isna(row['anios_experiencia']):
        motivos.append("experiencia_vacia")
    return ",".join(motivos)

rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)

In [11]:
#Exportar archivo curated
corredores.to_csv("corredores_curated.csv", index=False)